In [4]:
import numpy as np

In [2]:
class Value:
  """Stores a single scalar value and its gradient"""

  def __init__(self, data, _children=(), _op=''):
    self.data = data
    self.grad = 0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op # Operation that produced this node

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(data = self.data + other.data, _children = (self, other), _op = '+')

    # Plus operation distributes the gradient to both operands.
    # In backpropagation, the gradient from the output (out.grad) is
    # equally passed back to both operands (self.grad and other.grad),
    # if out = self + other
    # than d(out)/d(self) = 1 and d(out)/d(other) = 1 for simple addition.
    def _backward():
      self.grad += out.grad
      other.grad += out.grad
    out._backward = _backward


    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(data = self.data * other.data, _children = (self, other), _op = '*')

    # Multiplication operation distributes the gradient to both operands.
    # In backpropagation, the gradient from the output (out.grad) is distributed
    # based on the product rule: d(out)/d(self) = other and d(out)/d(other) = self.
    # Therefore, the gradient passed back to each operand is:
    # - self.grad += out.grad * other (derivative with respect to self)
    # - other.grad += out.grad * self (derivative with respect to other)
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward

    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), 'only supporting int/float powers for now'
    out = Value(self.data ** other, (self, ), f'**{other}')

    def _backward():
      self.grad += (other * self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out

  def relu(self):
    out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

    def _backward():
      self.grad += (self.data > 0) * out.grad
    out._backward = _backward

    return out

  def backward(self):
    # topological order all of the children in the graph
    topo = []
    visited = set()
    def build_topo(v):
      visited.add(v)
      for child in v._prev:
        build_topo(child)
      topo.append(v)
    build_topo(self)

    self.grad = 1

    for v in reversed(topo):
      v._backward()

  def __neg__(self): # -self
      return self * -1

  def __radd__(self, other): # other + self
      return self + other

  def __sub__(self, other): # self - other
      return self + (-other)

  def __rsub__(self, other): # other - self
      return other + (-self)

  def __rmul__(self, other): # other * self
      return self * other

  def __truediv__(self, other): # self / other
      return self * other**-1

  def __rtruediv__(self, other): # other / self
      return other * self**-1

  def __repr__(self):
      return f"Value(data={self.data}, grad={self.grad})"

  def zero_(self):
    self.grad = 0

In [3]:
class Module:
  def zero_grad(self):
    for p in self.parameters():
      p.grad = 0

  def parameters(self):
    return []

In [5]:
class Neuron(Module):

  def __init__(self, number_of_inputs, nonlin = True):
    self.weights = [Value(np.random.uniform(-1, 1)) for _ in range(number_of_inputs)]
    self.bias = Value(0)
    self.nonlin = nonlin

  def __call__(self, x):
    activation = sum((wi*xi for wi, xi in zip(self.weights, x)), self.bias)
    return activation.relu() if self.nonlin else activation

  def parameters(self):
    return self.weights + [self.bias]

  def __repr__(self):
    return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"

In [8]:
class Layer(Module):

  def __init__(self, number_of_inputs, number_of_outputs, **kwargs):
    self.neurons = [Neuron(number_of_inputs, **kwargs) for _ in range(number_of_outputs)]

  def __call__(self, x):
    out = [n(x) for n in self.neurons]
    return out[0] if len(out) == 1 else out

  def parameters(self):
    return [p for n in self.neurons for p in n.parameters()]

  def __repr__(self):
    return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

In [7]:
class MLP(Module):

  def __init__(self, number_of_inputs, number_of_outputs):
    size = [number_of_inputs] + number_of_outputs
    self.layers = [Layer(size[i], size[i + 1], nonlin=i!=len(number_of_outputs)-1) for i in range(len(number_of_outputs))]

  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

  def __repr__(self):
    return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"